# Instrumental Variables with DML: The PLIV Model

**Goal:** Estimate causal effects with an endogenous treatment using `DoubleMLPLIV`, and compare to biased OLS/PLR.

**Time:** 15 minutes

You will simulate endogeneity, show OLS and PLR fail, then use PLIV to recover the true effect.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from doubleml import DoubleMLData, DoubleMLPLIV, DoubleMLPLR
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Simulate: Endogenous Shipping Volume

Weather anomalies affect shipping volume but not commodity price spreads directly.
Market sentiment (unobserved) confounds the shipping-price relationship.

In [ ]:
n = 3000
p = 20
true_theta = 0.5

X = np.random.randn(n, p)
col_names = [f'X{i}' for i in range(p)]

Z = 0.5 * X[:, 0] + np.random.randn(n)  # Instrument: weather
U = np.random.randn(n)  # Unobserved: market sentiment

D = 0.6 * Z + 0.3 * X[:, 1] + 0.4 * U + np.random.randn(n) * 0.3
Y = (true_theta * D + 0.5 * X[:, 0] + 0.3 * X[:, 2]
     + 0.5 * U + np.random.randn(n) * 0.3)

print(f'True effect:  {true_theta}')
print(f'Corr(D, U):   {np.corrcoef(D, U)[0,1]:.3f} -- endogenous!')
print(f'Corr(Z, U):   {np.corrcoef(Z, U)[0,1]:.3f} -- instrument exogenous')
print(f'Corr(Z, D):   {np.corrcoef(Z, D)[0,1]:.3f} -- instrument relevant')

## Show OLS and PLR Are Biased

Both fail because they cannot control for the unobserved confounder U.

In [ ]:
# Naive OLS
ols = sm.OLS(Y, sm.add_constant(np.column_stack([D, X]))).fit()

# PLR (ignores endogeneity)
df = pd.DataFrame(X, columns=col_names)
df['price_spread'] = Y
df['shipping_volume'] = D
df['weather'] = Z

dml_data_no_iv = DoubleMLData(df, y_col='price_spread',
                               d_cols='shipping_volume', x_cols=col_names)
plr = DoubleMLPLR(dml_data_no_iv,
    ml_l=RandomForestRegressor(200, random_state=42),
    ml_m=RandomForestRegressor(200, random_state=42), n_folds=5)
plr.fit()

print(f'True effect:  {true_theta:.2f}')
print(f'OLS:          {ols.params[1]:.2f}  (biased by endogeneity)')
print(f'PLR:          {plr.coef[0]:.2f}  (biased -- U unobserved)')

## PLIV: Using the Weather Instrument

Now use weather as an instrument to handle the endogeneity.

In [ ]:
dml_data_iv = DoubleMLData(df, y_col='price_spread',
                            d_cols='shipping_volume',
                            x_cols=col_names,
                            z_cols='weather')

pliv = DoubleMLPLIV(dml_data_iv,
    ml_l=RandomForestRegressor(200, random_state=42),
    ml_m=RandomForestRegressor(200, random_state=42),
    ml_r=RandomForestRegressor(200, random_state=42),
    n_folds=5)
pliv.fit()

print('PLIV Results:')
print(pliv.summary)
print(f'\nTrue:  {true_theta:.2f}')
print(f'PLIV:  {pliv.coef[0]:.2f}')
print(f'OLS:   {ols.params[1]:.2f}')
print(f'PLR:   {plr.coef[0]:.2f}')

## Instrument Strength Diagnostic

Check the partial R-squared of the instrument.

In [ ]:
rf = RandomForestRegressor(200, random_state=42)
D_hat_with_Z = cross_val_predict(rf, np.column_stack([X, Z.reshape(-1, 1)]), D, cv=5)
D_hat_no_Z = cross_val_predict(rf, X, D, cv=5)

r2_with = r2_score(D, D_hat_with_Z)
r2_without = r2_score(D, D_hat_no_Z)
partial_r2 = (r2_with - r2_without) / (1 - r2_without)

print(f'R2 with instrument:    {r2_with:.3f}')
print(f'R2 without instrument: {r2_without:.3f}')
print(f'Partial R2:            {partial_r2:.3f}')
print(f'Instrument strength:   {"STRONG" if partial_r2 > 0.05 else "WEAK"}')

## Summary

**What you learned:**
1. When treatment is endogenous (unobserved confounders), OLS and PLR are biased
2. PLIV uses an instrument to identify the causal effect despite endogeneity
3. `DoubleMLPLIV` adds `z_cols` and `ml_r` for the instrument equation
4. Always check instrument strength via partial R-squared

**What is next:**
- **Module 08:** Heterogeneous treatment effects with `econml`
- **Module 09:** Production pipeline with IV diagnostics

**Key takeaway:** When you suspect endogeneity, you need an instrument.
PLIV combines IV identification with ML flexibility.